# Part 3 — Sales Forecasting

**Objective:** Forecast daily Revenue and COGS for 2023-01-01 → 2024-07-01 (548 days).

**Pipeline:**
1. Load ensemble model predictions
2. Calibrate COGS using historical ratios from  (2018–2022), grouped by 
3. Blend 60% historical ratio + 40% model ratio — corrects systematic COGS bias
4. Write 

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

_here = Path.cwd()
ROOT = next(
    (p for p in [_here, _here.parent, _here.parent.parent, _here.parent.parent.parent]
     if (p / "dataset").exists()),
    None,
)
assert ROOT is not None, f"Cannot find dataset/ from {_here}"

DATASET     = ROOT / "dataset"
SUBMISSIONS = ROOT / "part3" / "submissions"
DELIVERABLES = ROOT / "deliverables"
DELIVERABLES.mkdir(exist_ok=True)

FLOOR = 50_000.0
HIST_W, MODEL_W = 0.60, 0.40
AUG_ODD_CAP = 0.99

print(f"ROOT: {ROOT}")

ROOT: c:\Users\marki\Downloads\coolsies\datathon v1.3


In [2]:
# Load ensemble model predictions (548 days)
pred = pd.read_csv(SUBMISSIONS / "base_predictions.csv", parse_dates=["Date"])
assert len(pred) == 548
assert list(pred.columns) == ["Date", "Revenue", "COGS"]
print(f"Predictions: {pred.Date.min().date()} -> {pred.Date.max().date()} ({len(pred)} rows)")
print(pred[["Revenue", "COGS"]].describe().round(0))

Predictions: 2023-01-01 -> 2024-07-01 (548 rows)
          Revenue        COGS
count       548.0       548.0
mean    4379096.0   3988635.0
std     1927880.0   1674033.0
min     1368647.0   1394192.0
25%     2776154.0   2607767.0
50%     4187694.0   3722427.0
75%     5553852.0   4952750.0
max    12417944.0  11794405.0


In [3]:
# Historical COGS/Revenue ratios from sales.csv (2018-2022)
sales = pd.read_csv(DATASET / "sales.csv", parse_dates=["Date"])
hist  = sales[sales["Date"].dt.year.between(2018, 2022)].copy()
hist["month"]    = hist["Date"].dt.month
hist["odd_year"] = hist["Date"].dt.year % 2 == 1

ratio_lookup = {}
for (m, is_odd), g in hist.groupby(["month", "odd_year"]):
    r = g["COGS"].sum() / max(g["Revenue"].sum(), 1.0)
    if int(m) == 8 and bool(is_odd):
        r = min(r, AUG_ODD_CAP)
    ratio_lookup[(int(m), bool(is_odd))] = float(r)

odd_fb  = float(hist.loc[hist["odd_year"],  "COGS"].sum() / hist.loc[hist["odd_year"],  "Revenue"].sum())
even_fb = float(hist.loc[~hist["odd_year"], "COGS"].sum() / hist.loc[~hist["odd_year"], "Revenue"].sum())
print(f"Historical ratio lookup: {len(ratio_lookup)} (month, odd_year) buckets")

Historical ratio lookup: 24 (month, odd_year) buckets


In [4]:
# Apply 60% historical / 40% model COGS blend
pred["month"]       = pred["Date"].dt.month
pred["odd_year"]    = pred["Date"].dt.year % 2 == 1
pred["model_ratio"] = pred["COGS"] / pred["Revenue"].clip(lower=1.0)
pred["hist_ratio"]  = pred.apply(
    lambda r: ratio_lookup.get((r["month"], r["odd_year"]), odd_fb if r["odd_year"] else even_fb),
    axis=1,
)
pred["blended"] = HIST_W * pred["hist_ratio"] + MODEL_W * pred["model_ratio"]

out = pd.DataFrame({
    "Date":    pred["Date"].dt.strftime("%Y-%m-%d"),
    "Revenue": pred["Revenue"].round(2),
    "COGS":    np.maximum(pred["Revenue"] * pred["blended"], FLOOR).round(2),
})

print(f"Revenue mean: {out.Revenue.mean():>12,.2f}")
print(f"COGS mean:    {out.COGS.mean():>12,.2f}")
print(f"COGS/Rev:     {out.COGS.mean() / out.Revenue.mean():.4f}")

Revenue mean: 4,379,096.50
COGS mean:    3,847,025.65
COGS/Rev:     0.8785


In [5]:
# Verify format and save
assert len(out) == 548
assert list(out.columns) == ["Date", "Revenue", "COGS"]
assert out["Date"].iloc[0]  == "2023-01-01"
assert out["Date"].iloc[-1] == "2024-07-01"

out_path = DELIVERABLES / "submission.csv"
out.to_csv(out_path, index=False)
print(f"Saved {len(out)} rows -> {out_path}")
out.head()

Saved 548 rows -> c:\Users\marki\Downloads\coolsies\datathon v1.3\deliverables\submission.csv


,Date,Revenue,COGS
0,2023-01-01,3593616.43,3191555.75
1,2023-01-02,1743501.49,1627768.77
2,2023-01-03,1479373.93,1313300.69
3,2023-01-04,1550252.77,1328236.02
4,2023-01-05,1525645.72,1299267.06
